# Libraries & Data Import

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('../data/clean_data.csv')
sns.set_style('whitegrid')
os.makedirs('../images', exist_ok=True)

In [ ]:
df.head()

# Risk Score

In [ ]:
feature_adoption_threshold = df['feature_adoption_rate'].quantile(0.2)
seat_utilization_threshold = df['seat_utilization'].quantile(0.2)
tickets_threshold = 5

cust_stats = pd.DataFrame({
    'customer_id': df['customer_id'],
    'mrr_segment': df['MRR_segment'],
    'monthly_recurring_revenue': df['monthly_recurring_revenue'],
    'has_failed_payment': (df['payment_status'] == 'Failed').astype(int),
    'has_low_feature_adoption_rate': (df['feature_adoption_rate'] <= feature_adoption_threshold).astype(int),
    'has_low_seat_utilization': (df['seat_utilization'] <= seat_utilization_threshold).astype(int),
    'has_many_support_tickets': (df['support_tickets_30d'] >= tickets_threshold).astype(int),
    'is_churned': df['is_churned']
}).groupby('customer_id').max().reset_index()

flag_cols = ['has_failed_payment', 'has_low_feature_adoption_rate',
             'has_low_seat_utilization', 'has_many_support_tickets']

cust_stats['risk_score'] = cust_stats[flag_cols].sum(axis=1)

In [ ]:
cust_stats.head()

# Risk Levels

In [ ]:
cust_stats['risk_level'] = pd.cut(
    cust_stats['risk_score'],
    bins=[-1, 0, 2, 4],
    labels=['Low', 'Medium', 'High']
)

In [ ]:
score_table = cust_stats.groupby('risk_score').agg(
    clients=('customer_id', 'count'),
    churned=('is_churned', 'sum'),
    churn_rate=('is_churned', 'mean')
).round(4)
score_table['lift'] = (score_table['churn_rate'] / cust_stats['is_churned'].mean()).round(1)
score_table.head()

In [ ]:
level_table = cust_stats.groupby('risk_level', observed=True).agg(
    clients=('customer_id', 'count'),
    churned=('is_churned', 'sum'),
    churn_rate=('is_churned', 'mean')
).round(4).reset_index()

level_table['share_of_clients'] = (level_table['clients'] / level_table['clients'].sum()).round(3)
level_table['share_of_churned'] = (level_table['churned'] / level_table['churned'].sum()).round(3)
level_table['lift'] = (level_table['churn_rate'] / cust_stats['is_churned'].mean()).round(3)

level_table.head()

In [ ]:
avg_rate = cust_stats['is_churned'].mean()

fig, ax = plt.subplots(figsize=(8, 5))

sns.barplot(x=score_table.index.astype(str), y=score_table['churn_rate'].values * 100, ax=ax, color='#4C72B0')
ax.bar_label(ax.containers[0], labels=[f'{r * 100:.1f}%\n(n={n:,})' for r, n in zip(score_table['churn_rate'], score_table['clients'])],
             padding=3, fontsize=9, bbox=dict(facecolor='white', edgecolor='none', pad=1))
ax.axhline(avg_rate * 100, color='#C44E52', linestyle='--', linewidth=1, zorder=1, label=f'Average churn rate ({avg_rate * 100:.1f}%)')
ax.margins(y=0.15)
ax.set_title('Churn Rate by Risk Score')
ax.set_xlabel('Risk score (number of risk flags)')
ax.set_ylabel('Churn rate, %')
ax.legend()

plt.tight_layout()
plt.savefig('../images/churn_by_risk_score.png', dpi=150)
plt.show()

In [ ]:
shares = level_table.set_index('risk_level')[['share_of_clients', 'share_of_churned']] * 100

ax = shares.plot(kind='bar', figsize=(8, 5), color=['#4C72B0', '#C44E52'], rot=0)

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3, fontsize=9)

ax.set_title('Share of Clients vs Share of Churned Clients by Risk Level')
ax.set_xlabel('Risk level')
ax.set_ylabel('Share, %')
ax.legend(['Share of clients', 'Share of churned clients'])

plt.tight_layout()
plt.savefig('../images/risk_levels.png', dpi=150)
plt.show()

# Priority List

In [ ]:
rate_by_score = cust_stats.groupby('risk_score')['is_churned'].mean()
rate_by_score

In [ ]:
active = cust_stats[cust_stats['is_churned'] == 0].copy()
active['churn_rate'] = active['risk_score'].map(rate_by_score)
active['expected_loss'] = active['churn_rate'] * active['monthly_recurring_revenue']

In [ ]:
flag_labels = {
    'has_failed_payment': 'Failed payment',
    'has_low_feature_adoption_rate': 'Low feature adoption',
    'has_low_seat_utilization': 'Low seat utilization',
    'has_many_support_tickets': 'Many support tickets'
}

active['risk_reasons'] = ''
for col, label in flag_labels.items():
    active['risk_reasons'] += np.where(active[col] == 1, label + '; ', '')
active['risk_reasons'] = active['risk_reasons'].str.rstrip('; ')

active['recommended_action'] = np.select(
    [active['has_failed_payment'] == 1,
     active['has_low_feature_adoption_rate'] == 1,
     active['has_low_seat_utilization'] == 1,
     active['has_many_support_tickets'] == 1],
    ['Contact the client to resolve the failed payment',
     'Offer training on key features',
     'Customer success call to activate unused seats',
     'Account manager reviews open support tickets'],
    default='Account manager check-in'
)

In [ ]:
list_columns = ['customer_id', 'mrr_segment', 'monthly_recurring_revenue', 'risk_score',
                'churn_rate', 'expected_loss', 'risk_reasons', 'recommended_action']

top20 = active.nlargest(20, 'expected_loss')[list_columns]
top20

In [ ]:
top20.to_csv('../data/priority_list_top20.csv', index=False)

In [ ]:
risk_level_summary = active.groupby('risk_level', observed=True).agg(
    clients=('customer_id', 'count'),
    mrr=('monthly_recurring_revenue', 'sum'),
    expected_loss=('expected_loss', 'sum')
).reset_index()

risk_level_summary['share_of_clients'] = (risk_level_summary['clients'] / risk_level_summary['clients'].sum()).round(3)
risk_level_summary['share_of_mrr'] = (risk_level_summary['mrr'] / risk_level_summary['mrr'].sum()).round(3)
risk_level_summary['share_of_expected_loss'] = (risk_level_summary['expected_loss'] / risk_level_summary['expected_loss'].sum()).round(3)

risk_level_summary

In [ ]:
loss_sorted = active['expected_loss'].sort_values(ascending=False)

concentration = pd.Series(
    {f'Top {k}': loss_sorted.head(k).sum() / loss_sorted.sum() * 100 for k in (20, 100, 500)},
    name='Share of total expected loss, %'
)

concentration.round(1)

In [ ]:
auto_segments = ['Mass', 'Economy', 'Business']

auto_list = (
    active[active['mrr_segment'].isin(auto_segments)]
    .sort_values(['churn_rate', 'monthly_recurring_revenue', 'customer_id'],
                 ascending=[False, False, True])
    .groupby('mrr_segment', observed=True)
    .head(5)
    .copy()
)

auto_list['mrr_segment'] = pd.Categorical(auto_list['mrr_segment'], categories=auto_segments, ordered=True)
auto_list = auto_list.sort_values(['mrr_segment', 'churn_rate', 'monthly_recurring_revenue', 'customer_id'],
                                  ascending=[True, False, False, True])
auto_list = auto_list[list_columns]
auto_list

In [ ]:
auto_list.to_csv('../data/priority_list_auto_segments.csv', index=False)